# Visualization Layer Preparation - Gold Layer

Prepares visualization-ready tables for the Streamlit app.

**Inputs:**
- `{catalog}.{gold_schema}.expansion_candidates_h3_enhanced`
- `{catalog}.{silver_schema}.existing_stores_h3`
- `{catalog}.{silver_schema}.pois_competitors`
- `{catalog}.{silver_schema}.isochrones_convenience`
- `{catalog}.{bronze_schema}.census_states`

**Outputs:**
- `{catalog}.{gold_schema}.viz_expansion_candidates` - Normalized scores (0-1)
- `{catalog}.{gold_schema}.viz_existing_stores`
- `{catalog}.{gold_schema}.viz_competitors`
- `{catalog}.{gold_schema}.viz_convenience`
- `{catalog}.{gold_schema}.viz_h3_grid` - H3-8 covering MA boundary

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, explode, lit, when
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "jdub_demo_aws")
dbutils.widgets.text("bronze_schema", "geo_bronze")
dbutils.widgets.text("silver_schema", "geo_silver")
dbutils.widgets.text("gold_schema", "geo_gold")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

print(f"Catalog: {catalog}")
print(f"Bronze: {bronze_schema}, Silver: {silver_schema}, Gold: {gold_schema}")

## 1. Generate H3 Grid Covering Massachusetts

In [ ]:
# Load Massachusetts boundary
ma_boundary = spark.table(f"{catalog}.{bronze_schema}.census_states").filter(
    (col("state_abbr") == "MA") | (col("state_fips") == "25")
)

# Generate H3-8 grid covering Massachusetts
viz_h3_grid = ma_boundary.select(
    explode(expr("h3_coverash3string(ST_AsBinary(geometry), 5)")).alias("coarse_h3")
).select(
    explode(expr("h3_tochildren(coarse_h3, 8)")).alias("h3_cell_id")
).distinct().withColumn(
    "geometry", expr("ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326)")
).withColumn(
    "center_lat", expr("ST_Y(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))")
).withColumn(
    "center_lon", expr("ST_X(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))")
)

print(f"Generated H3-8 grid with {viz_h3_grid.count()} cells")

# Write H3 grid
viz_h3_grid.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_h3_grid")
print(f"Written to {catalog}.{gold_schema}.viz_h3_grid")

## 2. Prepare Expansion Candidates (Normalized Scores)

In [ ]:
# Load expansion candidates
candidates = spark.table(f"{catalog}.{gold_schema}.expansion_candidates_h3_enhanced")

# Calculate min/max for normalization
stats = candidates.agg(
    F.min("predicted_annual_sales").alias("min_sales"),
    F.max("predicted_annual_sales").alias("max_sales"),
    F.min("population").alias("min_pop"),
    F.max("population").alias("max_pop")
).collect()[0]

min_sales, max_sales = stats["min_sales"], stats["max_sales"]
min_pop, max_pop = stats["min_pop"], stats["max_pop"]

# Add normalized scores (0-1)
# Handle division by zero case
if max_sales > min_sales:
    viz_candidates = candidates.withColumn(
        "normalized_sales_score",
        (col("predicted_annual_sales") - lit(min_sales)) / lit(max_sales - min_sales)
    )
else:
    viz_candidates = candidates.withColumn("normalized_sales_score", lit(0.5))

if max_pop > min_pop:
    viz_candidates = viz_candidates.withColumn(
        "normalized_pop_score",
        (col("population") - lit(min_pop)) / lit(max_pop - min_pop)
    )
else:
    viz_candidates = viz_candidates.withColumn("normalized_pop_score", lit(0.5))

viz_candidates = viz_candidates.withColumn(
    "percentile_rank",
    F.percent_rank().over(Window.orderBy("predicted_annual_sales"))
).withColumn(
    "geometry", expr("ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326)")
)

print(f"Prepared {viz_candidates.count()} visualization candidates")

# Write
viz_candidates.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_expansion_candidates")
print(f"Written to {catalog}.{gold_schema}.viz_expansion_candidates")

## 3. Prepare Existing Stores

In [ ]:
try:
    existing_stores = spark.table(f"{catalog}.{silver_schema}.existing_stores_h3")
    
    viz_existing = existing_stores.select(
        "store_number",
        "latitude",
        "longitude",
        "store_type",
        "city",
        "state",
        "population",
        col("total_poi_count").alias("poi_count") if "total_poi_count" in existing_stores.columns else lit(0).alias("poi_count"),
        "geometry"
    ).withColumn(
        "marker_type", lit("existing_lce")
    )
    
    print(f"Prepared {viz_existing.count()} existing stores")
    
    viz_existing.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_existing_stores")
    print(f"Written to {catalog}.{gold_schema}.viz_existing_stores")
    
except Exception as e:
    print(f"Could not prepare existing stores: {e}")

## 4. Prepare Competitors

In [ ]:
try:
    competitors = spark.table(f"{catalog}.{silver_schema}.pois_competitors")
    
    viz_competitors = competitors.select(
        col("poi_id").alias("id"),
        "name",
        "latitude",
        "longitude",
        "poi_category",
        "poi_subcategory",
        "address"
    ).withColumn(
        "marker_type", lit("competitor")
    ).withColumn(
        "geometry", expr("ST_Point(longitude, latitude)")
    )
    
    print(f"Prepared {viz_competitors.count()} competitors")
    
    viz_competitors.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_competitors")
    print(f"Written to {catalog}.{gold_schema}.viz_competitors")
    
except Exception as e:
    print(f"Could not prepare competitors: {e}")

## 5. Prepare Convenience Store Isochrones

In [ ]:
try:
    convenience = spark.table(f"{catalog}.{silver_schema}.isochrones_convenience")
    
    viz_convenience = convenience.select(
        col("location_id").alias("id"),
        "store_type",
        "latitude",
        "longitude",
        "city",
        "state",
        "drive_time_minutes",
        "area_sqkm",
        "geometry"
    ).withColumn(
        "marker_type", lit("convenience")
    )
    
    print(f"Prepared {viz_convenience.count()} convenience store isochrones")
    
    viz_convenience.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_convenience")
    print(f"Written to {catalog}.{gold_schema}.viz_convenience")
    
except Exception as e:
    print(f"Could not prepare convenience isochrones: {e}")

## Summary

In [ ]:
print("=" * 60)
print("VISUALIZATION LAYER SUMMARY")
print("=" * 60)

viz_tables = [
    "viz_h3_grid",
    "viz_expansion_candidates",
    "viz_existing_stores",
    "viz_competitors",
    "viz_convenience"
]

for table_name in viz_tables:
    try:
        count = spark.table(f"{catalog}.{gold_schema}.{table_name}").count()
        print(f"  {table_name}: {count:,} rows")
    except Exception as e:
        print(f"  {table_name}: NOT AVAILABLE")

print("\n" + "=" * 60)
print("VISUALIZATION LAYER COMPLETE")
print("=" * 60)

## Folium Map Visualization with Marker Clustering

Interactive map showing:
- **LCE Stores** (green markers with clustering)
- **LCE Trade Areas** (orange isochrones)
- **Convenience Store Trade Areas** (blue isochrones)
- **Expansion Candidates** (gold/orange markers)

Uses Little Caesars branding colors and marker clustering for performance.

In [ ]:
%pip install folium --quiet

import folium
from folium.plugins import MarkerCluster, Fullscreen
import pandas as pd
from shapely import wkt
from shapely.geometry import mapping

print("Loading data for Folium map...")

# Load LCE stores
lce_stores_pd = (
    spark.table(f"{catalog}.{gold_schema}.viz_existing_stores")
    .select("store_number", "latitude", "longitude", "city", "state", "population")
    .toPandas()
)

# Load LCE isochrones as GeoJSON
lce_isochrones_df = spark.table(f"{catalog}.{silver_schema}.isochrones_lce")
lce_isochrones_gdf = (
    lce_isochrones_df
    .selectExpr(
        "location_id as store_number",
        "ST_AsText(geometry) as geometry_wkt",
        "drive_time_minutes",
        "area_sqkm"
    )
    .toPandas()
)

lce_isochrones_geojson = {
    "type": "FeatureCollection",
    "features": []
}

for _, row in lce_isochrones_gdf.iterrows():
    geom = wkt.loads(row['geometry_wkt'])
    feature = {
        "type": "Feature",
        "properties": {
            "store_number": row['store_number'],
            "drive_time_minutes": row['drive_time_minutes'],
            "area_sqkm": row['area_sqkm']
        },
        "geometry": mapping(geom)
    }
    lce_isochrones_geojson["features"].append(feature)

# Load convenience store isochrones as GeoJSON
convenience_isochrones_df = spark.table(f"{catalog}.{silver_schema}.isochrones_convenience")
convenience_isochrones_gdf = (
    convenience_isochrones_df
    .selectExpr(
        "location_id",
        "ST_AsText(geometry) as geometry_wkt",
        "drive_time_minutes",
        "area_sqkm"
    )
    .toPandas()
)

convenience_isochrones_geojson = {
    "type": "FeatureCollection",
    "features": []
}

for _, row in convenience_isochrones_gdf.iterrows():
    geom = wkt.loads(row['geometry_wkt'])
    feature = {
        "type": "Feature",
        "properties": {
            "location_id": row['location_id'],
            "drive_time_minutes": row['drive_time_minutes'],
            "area_sqkm": row['area_sqkm']
        },
        "geometry": mapping(geom)
    }
    convenience_isochrones_geojson["features"].append(feature)

# Load expansion candidates
candidates_pd = (
    spark.table(f"{catalog}.{gold_schema}.viz_expansion_candidates")
    .orderBy(F.desc("predicted_annual_sales"))
    .limit(20)  # Top 20 candidates
    .select("h3_cell_id", "latitude", "longitude", "predicted_annual_sales", "population")
    .toPandas()
)

print(f"\nLoaded data:")
print(f"  LCE stores: {len(lce_stores_pd)}")
print(f"  LCE trade areas: {len(lce_isochrones_geojson['features'])}")
print(f"  Convenience trade areas: {len(convenience_isochrones_geojson['features'])}")
print(f"  Expansion candidates: {len(candidates_pd)}")

In [ ]:
# Create Folium map centered on Massachusetts
ma_center = [42.4072, -71.3824]
folium_map = folium.Map(
    location=ma_center,
    zoom_start=9,
    tiles='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',
    attr='CartoDB'
)

print("Building map layers...")

# Layer 1: LCE Trade Areas (orange isochrones)
if lce_isochrones_geojson['features']:
    for feature in lce_isochrones_geojson['features']:
        coords = feature['geometry']['coordinates'][0]
        polygon_coords = [[lat, lon] for lon, lat in coords]
        
        folium.Polygon(
            locations=polygon_coords,
            color='#FF8C00',  # Orange stroke
            fill=True,
            fillColor='#FF8C00',
            fillOpacity=0.1,
            weight=1,
            popup=f"<b>LCE Store {feature['properties']['store_number']}</b><br>"
                  f"Drive time: {feature['properties']['drive_time_minutes']} min<br>"
                  f"Area: {feature['properties']['area_sqkm']:.2f} km²"
        ).add_to(folium_map)
    
    print(f"  Added {len(lce_isochrones_geojson['features'])} LCE trade areas")

# Layer 2: Convenience Trade Areas (blue isochrones)
if convenience_isochrones_geojson['features']:
    for feature in convenience_isochrones_geojson['features']:
        coords = feature['geometry']['coordinates'][0]
        polygon_coords = [[lat, lon] for lon, lat in coords]
        
        folium.Polygon(
            locations=polygon_coords,
            color='#3b82f6',  # Blue stroke
            fill=True,
            fillColor='#3b82f6',
            fillOpacity=0.08,
            weight=1,
            popup=f"<b>Convenience Store</b><br>"
                  f"Location: {feature['properties']['location_id']}<br>"
                  f"Drive time: {feature['properties']['drive_time_minutes']} min<br>"
                  f"Area: {feature['properties']['area_sqkm']:.2f} km²"
        ).add_to(folium_map)
    
    print(f"  Added {len(convenience_isochrones_geojson['features'])} convenience trade areas")

# Layer 3: LCE Store Markers with GREEN clustering
lce_cluster = MarkerCluster(
    name='LCE Stores',
    options={
        'maxClusterRadius': 50,
        'iconCreateFunction': '''
            function(cluster) {
                return L.divIcon({
                    html: '<div style="background-color: #10b981; color: white; border-radius: 50%; width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; font-weight: bold; border: 3px solid #059669;"><span>' + cluster.getChildCount() + '</span></div>',
                    className: 'marker-cluster',
                    iconSize: L.point(40, 40)
                });
            }
        '''
    }
).add_to(folium_map)

for _, store in lce_stores_pd.iterrows():
    folium.CircleMarker(
        location=[store['latitude'], store['longitude']],
        radius=8,
        popup=f"<b>Little Caesars</b><br>"
              f"Store: {store['store_number']}<br>"
              f"Location: {store.get('city', 'N/A')}, {store.get('state', 'N/A')}<br>"
              f"Population: {store.get('population', 0):,.0f}",
        tooltip=f"LCE Store {store['store_number']}",
        color='#10b981',
        fill=True,
        fillColor='#34d399',
        fillOpacity=0.8,
        weight=2
    ).add_to(lce_cluster)

print(f"  Added {len(lce_stores_pd)} LCE store markers with clustering")

# Layer 4: Expansion Candidates (gold/orange markers)
for _, candidate in candidates_pd.iterrows():
    folium.CircleMarker(
        location=[candidate['latitude'], candidate['longitude']],
        radius=8,
        popup=f"<b>Expansion Candidate</b><br>"
              f"H3 Cell: {candidate['h3_cell_id']}<br>"
              f"Predicted Sales: ${candidate['predicted_annual_sales']:,.0f}<br>"
              f"Population: {candidate['population']:,.0f}",
        tooltip=f"Expansion: ${candidate['predicted_annual_sales']:,.0f}",
        color='#f59e0b',
        fill=True,
        fillColor='#fbbf24',
        fillOpacity=0.8,
        weight=2
    ).add_to(folium_map)

print(f"  Added {len(candidates_pd)} expansion candidate markers")

print("\nMap layers complete!")

In [ ]:
# Add legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 300px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 15px; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.3);">
<p style="margin-bottom: 12px; font-weight: bold; font-size: 16px; color: #1a1a1a;">LCE Hunger Detection Map</p>
<p style="margin: 8px 0;">
    <span style="display: inline-block; width: 20px; height: 20px; 
                 background-color: #10b981; border-radius: 50%; border: 2px solid #059669;
                 vertical-align: middle; margin-right: 8px;"></span>
    <strong>LCE Stores</strong> (clustered)
</p>
<p style="margin: 8px 0;">
    <span style="display: inline-block; width: 40px; height: 12px; 
                 background-color: rgba(255,140,0,0.1); 
                 border: 2px solid #FF8C00; 
                 vertical-align: middle; margin-right: 8px;"></span>
    LCE Trade Areas (5-min)
</p>
<p style="margin: 8px 0;">
    <span style="display: inline-block; width: 40px; height: 12px; 
                 background-color: rgba(59,130,246,0.08); 
                 border: 2px solid #3b82f6; 
                 vertical-align: middle; margin-right: 8px;"></span>
    Convenience Trade Areas
</p>
<p style="margin: 8px 0;">
    <span style="display: inline-block; width: 20px; height: 20px; 
                 background-color: #fbbf24; border-radius: 50%; border: 2px solid #f59e0b;
                 vertical-align: middle; margin-right: 8px;"></span>
    Expansion Candidates
</p>
<p style="margin-top: 12px; padding-top: 8px; border-top: 1px solid #ddd; font-size: 11px; color: #666;">
    <strong>Tip:</strong> Zoom in/out to see clusters split/merge
</p>
</div>
'''

folium_map.get_root().html.add_child(folium.Element(legend_html))

# Add fullscreen button
Fullscreen(
    position='topleft',
    title='Expand map',
    title_cancel='Exit fullscreen',
    force_separate_button=True
).add_to(folium_map)

# Display the map
folium_map